In [ ]:
import importlib

if importlib.util.find_spec("easyocr") is None:
    !pip install docling langchain-docling langchain-community PyMuPDF flask easyocr
else:
    print("Packages already installed, skipping...")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from langchain_docling.loader import DoclingLoader, ExportType
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorDevice, EasyOcrOptions, AcceleratorOptions
from docling.document_converter import PdfFormatOption

from langchain_community.document_loaders import PyMuPDFLoader

from pathlib import Path
from flask import Flask, jsonify

import threading
import pymupdf as fitz
import time
import json
import gc
import re

# Declare variables
SIGNATURE_KEYWORDS = ["Người ký", "Ký bởi", "Ký:", "Cơ quan:", "Email:", "Thời gian ký"]
BASE_PATH = "/content/drive/MyDrive/Legal_RAG_assistant/legal_docs"
PROCESSED_PATH = "/content/drive/MyDrive/Legal_RAG_assistant/processed"

HEADER_PATTERNS = {
    'phần': re.compile(
        r'^\s*#*\s*Ph[aầ]n\s+(th[uứ]\s+\w+|[IVXLCDM\d]+)',
        re.IGNORECASE
    ),
    'chương': re.compile(
        r'^\s*#*\s*Ch[uươ][oơ]ng\s+[IVXLCDM\d]+',
        re.IGNORECASE
    ),
    'mục': re.compile(
        r'^\s*#*\s*M[uụ]c\s+\d+',
        re.IGNORECASE
    ),
    'tiểu_mục': re.compile(
        r'^\s*#*\s*Ti[eể]u\s+m[uụ]c\s+\d+',
        re.IGNORECASE
    ),
    'điều': re.compile(
        r'^\s*#*\s*[ĐđÐð]i[eề][uụ]\s+\d+[\.\:]?\s',
        re.IGNORECASE
    ),
}
LEVEL_HIERARCHY = ['phần', 'chương', 'mục', 'tiểu_mục', 'điều']

# Init
pipeline_options = PdfPipelineOptions(
    allow_external_plugins=True,
    do_ocr=True,
    do_table_structure=False,
    accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA)
)

pipeline_options.ocr_options = EasyOcrOptions(
    lang=["vi","en"],
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

def is_text_extractable(docs, threshold=200):
    if not docs:
        return False
    avg_chars = sum(len(d.page_content) for d in docs) / len(docs)
    return avg_chars > threshold

def remove_signature_block_from_vietnamese_documents(file_path: str) -> str:
    tmp_path = file_path + ".tmp"
    doc = fitz.open(file_path)

    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if b["type"] != 0:
                continue
            text = b['lines'][0]['spans'][0]['text']
            if any(kw in text for kw in SIGNATURE_KEYWORDS):
                page.add_redact_annot(fitz.Rect(b['bbox']), fill=(1,1,1))
        page.apply_redactions()

    doc.save(tmp_path)
    doc.close()

    Path(tmp_path).replace(file_path)
    return file_path

def load_and_process_docs(file_path):
    cleaned_file_path = remove_signature_block_from_vietnamese_documents(file_path)

    print(f"-> [TIMING]: Quick text check on {cleaned_file_path}...")
    start_pymupdf = time.perf_counter()

    try:
        loader = PyMuPDFLoader(cleaned_file_path)
        sample_docs = list(loader.load())[:10]
    except Exception as e:
        print(f"PyMuPDF failed: {e}")
        sample_docs = []

    end_pymupdf = time.perf_counter()

    if is_text_extractable(sample_docs):
        print(f"-> [load_and_process_docs]: Native PDF detected (checked in {end_pymupdf - start_pymupdf:.4f}s). Loading full doc...")
        full_loader = PyMuPDFLoader(cleaned_file_path)
        return full_loader.load()

    # Fallback: Using OCR for scanned PDF file
    docs = []
    try:
        print(f"-> [load_and_process_docs]: Scanned PDF detected. Starting Docling OCR on {cleaned_file_path}. This may take a while...")
        start_ocr = time.perf_counter()

        loader = DoclingLoader(cleaned_file_path, converter=converter, export_type=ExportType.MARKDOWN)
        docs = list(loader.lazy_load())

        end_ocr = time.perf_counter()
        print(f"-> [load_and_process_docs]: Docling OCR finished in {end_ocr - start_ocr:.4f} seconds.")
    except Exception as e:
        print(f"-> [load_and_process_docs]: Error loading document(s) from {cleaned_file_path}: {e}")

    return docs

def classify_block(block: str) -> str:
    for name, pattern in HEADER_PATTERNS.items():
        if pattern.match(block):
            return name
    return "content"

def detect_header_levels(blocks: list) -> list:
    """Scan all blocks, return which levels exist in order."""
    found = set()
    for block in blocks:
        block_type = classify_block(block)
        if block_type in LEVEL_HIERARCHY:
            found.add(block_type)

    # Return in hierarchy order, top to bottom
    return [l for l in LEVEL_HIERARCHY if l in found]

def detect_document_separators(full_text: str) -> list:
    """Detect separator and split accordingly."""
    if '\n\n' in full_text:
        blocks = full_text.split('\n\n')
    else:
        blocks = full_text.split('\n')
    return [b.strip() for b in blocks if b.strip()]

def strip_markdown(text: str) -> str:
    return re.sub(r'^#+\s*', '', text).strip()

def split_blocks_by_level(blocks, level):
    """Split blocks at every occurrence of `level` header."""
    sections = []
    current = {'title': None, 'blocks': []}
    before = []  # blocks before first header

    for block in blocks:
        if classify_block(block) == level:
            if current['title']:
                sections.append(current)
            current = {'title': strip_markdown(block), 'blocks': []}
        elif current['title']:
            current['blocks'].append(block)
        else:
            before.append(strip_markdown(block))

    if current['title']:
        sections.append(current)

    return before, sections

def extract_chunks_from_document(document):
    full_text = "\n".join(p.page_content for p in document)
    print(repr(full_text[:500]))
    blocks = detect_document_separators(full_text)

    for i, block in enumerate(blocks[:10]):
        block_type = classify_block(block)
        print(f"[{i}] type={block_type} | {block[:60]}")

    levels = detect_header_levels(blocks)
    child_level = levels[-1] if levels else None

    parent_level = levels[-2] if len(levels) >= 2 else None

    # Split by parent
    preamble_lines, parent_sections = split_blocks_by_level(blocks, parent_level)

    # Split each parent by child
    chunks = []
    for section in parent_sections:
        _, children = split_blocks_by_level(section['blocks'], child_level)

        chunks.append({
            'parent': section['title'],
            'parent_content': '\n'.join(
                strip_markdown(b) for b in section['blocks']
            ),
            'children': [
                {
                    'title': c['title'],
                    'content': '\n'.join(strip_markdown(b) for b in c['blocks']),
                }
                for c in children
            ],
        })

    preamble = '\n'.join(preamble_lines)
    return preamble, chunks

# Process each folder
for country_folder in Path(BASE_PATH).iterdir():
    if not country_folder.is_dir():
        continue
    processed_country_folder = Path(PROCESSED_PATH) / country_folder.name
    processed_country_folder.mkdir(parents=True, exist_ok=True)

    # Process each pdf files
    for pdf_file in country_folder.glob("*.pdf"):
        output_file = processed_country_folder / f"{pdf_file.stem}.json"

        if output_file.exists():
            print(f"-> [load_and_process_docs]: Skipping {pdf_file.name} already processed")
            continue
        document = load_and_process_docs(str(pdf_file))
        preample, chunks = extract_chunks_from_document(document)

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump({
                  "jurisdiction": country_folder.name,
                  "domain": pdf_file.stem.replace("_", " "),
                  "source": pdf_file.name,
                  "preample": preample,
                  "chunks": chunks
              }, f, ensure_ascii=False, indent=2)
        print(f"-> [load_and_process_docs]: {pdf_file.name} is saved")

def ocr_evaluation(pdf_path: str, ground_truth: list[str], num_pages: int = 5):
    pages = load_and_process_docs(pdf_path)
    for i, p in enumerate(pages[:5]):
        print(f"--- chunk {i} ---")
        print(p.metadata)
        print(p.page_content[:200])
    easyocr_texts = [p.page_content for p in pages[:num_pages]]

    def compute_cer(ref, hyp):
        ref, hyp = ref.strip(), hyp.strip()
        if not ref:
            return 0.0 if not hyp else 1.0
        m, n = len(ref), len(hyp)
        dp = list(range(n + 1))
        for i in range(1, m + 1):
            prev, dp[0] = dp[:], i
            for j in range(1, n + 1):
                dp[j] = prev[j-1] if ref[i-1] == hyp[j-1] else 1 + min(prev[j], dp[j-1], prev[j-1])
        return dp[n] / m

    print("=" * 60)
    cer_scores = []
    ground_truth = []       # Create a custom one to evaluate OCR
    for i in range(num_pages):
        ref = ground_truth[i] if i < len(ground_truth) else ""
        hyp = easyocr_texts[i] if i < len(easyocr_texts) else ""
        cer = compute_cer(ref, hyp)
        cer_scores.append(cer)

        status = "✓ Good" if cer < 0.1 else ("⚠ Average" if cer < 0.3 else "✗ Bad")
        print(f"Trang {i+1}: CER = {cer:.2%}  {status}")
        print(f"  [Ground truth] {ref[:150]!r}")
        print(f"  [EasyOCR]      {hyp[:150]!r}\n")


    avg_cer = sum(cer_scores) / len(cer_scores)
    print(f"Average CER: {avg_cer:.2%}")
    print("=" * 60)
    return cer_scores

app = Flask(__name__)

@app.route("/ingest", methods=["GET"])
def ingest():
    try:
        result = []
        for json_file in Path(PROCESSED_PATH).rglob("*.json"):
            with open(json_file, "r", encoding="utf-8") as f:
                result.append(json.load(f))

        print(f"-> [ingest]: Serving {len(result)} documents")
        return jsonify({"status": "ok", "documents": result})
    except Exception as e:
        print(f"-> [ingest]: Ingest failed: {e}")
        return jsonify({"status": "error", "message": str(e)})

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

print("Flask server starting up...")
threading.Thread(target=lambda: app.run(port=5000, debug=False, use_reloader=False)).start()
print("Flask server running on port 5000")

In [ ]:
import subprocess

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start the tunnel and capture the public URL
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait until the public URL appears in the logs
for line in proc.stdout:
    print(line.strip())
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\nYOUR PUBLIC URL: {public_url}")
        print(f"Copy this into your local app!")
        break